<a href="https://colab.research.google.com/github/nitik1998/GENIE_DiffusionLearning/blob/main/notebooks/Task3_Diffusion_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GENIE Task 3 — Diffusion Models

This notebook is the mentor-facing Colab wrapper for Task 3.
It uses `src/task3_diffusion.py` as the source of truth.

**Two experiment modes:**
- `latent_diffusion` **(main)** — DDPM on 256-dim latents from a frozen Task 1 VAE. Fast to train.
- `image_ddpm` **(baseline)** — Pixel-space DDPM with U-Net on raw 3×125×125 jet images.

Set `DIFFUSION_MODE` and `RUN_MODE` in Cell 3 before running.

In [ ]:
# 1. Optional Google Drive mount
from pathlib import Path

USE_GOOGLE_DRIVE = True
DRIVE_MOUNT_POINT = Path('/content/drive')

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT_POINT))
        print(f'Mounted Google Drive at {DRIVE_MOUNT_POINT}')
    except ImportError:
        print('google.colab is not available in this environment.')
else:
    print('Skipping Google Drive mount.')

In [ ]:
# 2. Clone repo and install dependencies
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nitik1998/GENIE_DiffusionLearning.git'
REPO_DIR = Path('/content/GENIE_DiffusionLearning')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Repo ready at: {REPO_DIR}')
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# 3. Runtime config — edit these two lines
DIFFUSION_MODE = "latent_diffusion"  # "latent_diffusion" or "image_ddpm"
RUN_MODE       = "full"              # "sanity" (3 epochs, 1000 events) or "full"

# ── Advanced settings (usually leave as-is) ──────────────────────────────────
SEED           = 42
FORCE_RERUN    = True
SANITY_EPOCHS  = 3
FULL_EPOCHS    = 30
SANITY_EVENTS  = 1_000

from pathlib import Path
import json, os, random, shlex, shutil, subprocess, sys, time
import torch
from IPython.display import Image, Markdown, display

from src.config import get_auto_batch_size

def detect_vram_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

EPOCHS     = SANITY_EPOCHS if RUN_MODE == "sanity" else FULL_EPOCHS
MAX_EVENTS = SANITY_EVENTS if RUN_MODE == "sanity" else None
EXP_NAME   = f"{DIFFUSION_MODE}_sanity" if RUN_MODE == "sanity" else DIFFUSION_MODE
BATCH_SIZE = int(get_auto_batch_size(task_num=3))
VRAM_GB    = round(detect_vram_gb(), 1)

DATASET_SOURCE = Path('/content/drive/MyDrive/quark-gluon_data-set_n139306.hdf5')
if not DATASET_SOURCE.exists():
    DATASET_SOURCE = Path('/content/quark-gluon_data-set_n139306.hdf5')

RESULTS_ROOT   = Path('/content/results_colab')
CHECKPOINT_ROOT = RESULTS_ROOT / 'checkpoints'
DATA_DIR        = REPO_DIR / 'data'
DATASET_TARGET  = DATA_DIR / 'quark-gluon_data-set_n139306.hdf5'
FINAL_RESULTS_DIR = RESULTS_ROOT / 'task3' / EXP_NAME

random.seed(SEED)
import numpy as np
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_DIR.mkdir(parents=True, exist_ok=True)
for path in [RESULTS_ROOT, CHECKPOINT_ROOT, FINAL_RESULTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if DATASET_SOURCE.exists() and DATASET_SOURCE.resolve() != DATASET_TARGET.resolve():
    shutil.copy2(DATASET_SOURCE, DATASET_TARGET)
elif not DATASET_TARGET.exists():
    raise FileNotFoundError(f"Dataset not found at {DATASET_SOURCE} or {DATASET_TARGET}")

os.environ['GENIE_PROJECT_ROOT']  = str(REPO_DIR)
os.environ['GENIE_DATA_DIR']      = str(DATA_DIR)
os.environ['GENIE_OUTPUT_DIR']    = str(RESULTS_ROOT)
os.environ['GENIE_CHECKPOINT_DIR']= str(CHECKPOINT_ROOT)

print(f"Python executable: {sys.executable}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM (GB): {VRAM_GB}")
print(f"Diffusion mode: {DIFFUSION_MODE}")
print(f"Run mode: {RUN_MODE}  /  Epochs: {EPOCHS}")
print(f"Experiment name: {EXP_NAME}")
print(f"Batch size: {BATCH_SIZE}")

def show_png(path, width=900):
    path = Path(path)
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

def print_metrics(path, title):
    path = Path(path)
    if not path.exists():
        print(f'Missing metrics file: {path}')
        return
    payload = json.loads(path.read_text())
    metrics = payload.get('metrics', payload)
    display(Markdown(f'### {title}'))
    for key, value in metrics.items():
        print(f'{key}: {value}')

In [ ]:
# 4. Run Task 3 — epoch logs stream live below
import subprocess, sys, time

cmd = [
    sys.executable, '-u',
    'src/task3_diffusion.py',
    '--mode',       DIFFUSION_MODE,
    '--exp-name',   EXP_NAME,
    '--epochs',     str(EPOCHS),
    '--batch-size', '0',   # 0 = auto-scale for H100
    '--seed',       str(SEED),
]
if MAX_EVENTS is not None:
    cmd += ['--max-events', str(MAX_EVENTS)]
if FORCE_RERUN:
    cmd += ['--force-rerun']

print(f"Running: {' '.join(cmd)}")
print("Epoch logs should stream live below.\n")

start_time = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
elapsed = time.time() - start_time
if proc.returncode != 0:
    raise RuntimeError(f'Task 3 command failed with exit code {proc.returncode}')
print(f"\nTask 3 run finished in {elapsed / 60:.1f} minutes")

In [ ]:
# 5. Display results
print_metrics(FINAL_RESULTS_DIR / 'metrics.json', f'Task 3 Metrics ({DIFFUSION_MODE})')

display(Markdown('### Training Curves'))
show_png(FINAL_RESULTS_DIR / 'training_curves.png')

display(Markdown('### Original vs Generated (channel-wise)'))
show_png(FINAL_RESULTS_DIR / 'original_vs_reconstructed.png')

display(Markdown('### Original vs Generated vs Abs Error'))
show_png(FINAL_RESULTS_DIR / 'original_vs_reconstructed_vs_abs_error.png')

In [ ]:
# 6. Download results zip
import shutil
from google.colab import files

zip_name = f'{EXP_NAME}.zip'
zip_path = f'/content/{zip_name}'
shutil.make_archive(f'/content/{EXP_NAME}', 'zip', str(FINAL_RESULTS_DIR.parent), str(FINAL_RESULTS_DIR.name))
print(f'Created: {zip_path}')
files.download(zip_path)